# 08 — Entrega
**Hackathon Databricks · Terra Signal · Grupo 4**

**Equipe:** André Ferreira, Antonio Menezes, Júlia Vale e Lucas Menghi

**O que este notebook faz:** gera o arquivo de previsão da entrega a partir das previsões que o 06 publicou, confere o
formato (D13 e D23) e junta os números da história para a apresentação.

| # | Pergunta de negócio | Etapa |
|---|---|---|
| E1 | O arquivo está no formato que a organização pede? | 1. Gerar e conferir o arquivo |
| E2 | Que história os números contam, do dado à ação? | 2. Resumo para a apresentação |

**De onde vem cada coisa:**

| Entrada | Notebook | O que traz |
|---|---|---|
| `gold_churn_inferencia_scored` | 06 | Probabilidade, previsão (0/1 no limiar do 06) e decil de risco de cada cliente da inferência |
| `gold_churn_modelo_info` | 06 | O modelo escolhido, o limiar e as métricas no teste |
| `gold_churn_abt_scored` | 06 | O mesmo para os 5.634 clientes do histórico, com a marcação treino/teste |
| `gold_churn_segmentos_perfil` | 07 | Segmentos, risco e receita em risco |
| `inference.csv` | `_comum` | A ordem oficial dos 1.409 clientes |

**Formato do arquivo:** `prediction_<equipe>_<timestamp>.csv`, com `customerID` e `prediction` (Yes/No), uma linha
por cliente, na ordem do `inference.csv`. **Conferir com o `NOTEBOOK.ipynb` oficial do repositório** (nomes das colunas,
Yes/No ou 1/0, com ou sem índice) antes de enviar: os parâmetros estão no início da 1.1.

In [ ]:
%run ./_comum

## Checklist de decisões

**A mesma tabela em todos os notebooks.** ☑ = decidido (com a data); ◐ = sugestão nossa, a confirmar (as de variáveis, na modelagem); ☐ = em aberto.

| # | Decisão | Opções | Proposta | Impacto | Evidência no 01 | No 02 | Depende de | Decisão do grupo |
|---|---|---|---|---|---|---|---|---|
| D1 | Grafia de `PhoneService` (70 + 23) | Padronizar | Padronizar para `Yes`/`No` | Sem isso, `yes` e `Yes` viram categorias diferentes no modelo | 4.1 | Aplicado | Carlos: só erro de padronização (respondido) | ☑ Padronizar para `Yes`/`No` (10/11) |
| D2 | `TotalCharges` nulo (278 + 64) | a) preencher com `MonthlyCharges` × `tenure`; b) excluir; c) deixar ausente | a | Excluir não serve para os 64 da inferência, que precisam de previsão, e perde o sinal da 4.3. A regra vale para 99% dos clientes | 4.1, 4.2, 4.3 | Aplicado | G2; Carlos: preencher assim é "razoável, desde que lembrem que é uma aproximação" (respondido); falta saber por que o valor está ausente | ☑ Preencher com `MonthlyCharges` × `tenure`, como aproximação (10/11) |
| D3 | `TotalCharges` nulo com `tenure` = 0 (10 + 1) | a) 0; b) outro valor | a (sai da própria regra: mensalidade × 0) | Clientes novos, sem cobrança ainda; efeito pequeno | 4.2 | Aplicado | G2 | ☑ 0 (10/11) |
| D4 | `tenure` = `unknown` (19 + 4) | a) `TotalCharges` ÷ `MonthlyCharges`, arredondado; b) mediana; c) excluir | a; mediana do histórico no único cliente sem as duas colunas | `tenure` é das variáveis mais ligadas ao churn: um erro aqui muda a faixa de tempo de casa do cliente | 4.1, 4.2 | Aplicado | G2; Carlos: 72 meses é teto real (respondido) | ☑ `TotalCharges` ÷ `MonthlyCharges`, arredondado e limitado a 72; mediana do histórico no caso sem os dois (10/11) |
| D5 | Indicador `fl_total_charges_ausente` | Criar ou não | Criar e comparar o modelo com e sem ele | Preserva o sinal da ausência (33,1% contra 26,4% de churn, p = 0,016) | 4.3 | Criado | Carlos: por que falta `TotalCharges` | ☑ Criar; o modelo testa com e sem (10/11) |
| D6 | Indicador `fl_tenure_ausente` | Criar ou não | Criar, mas tende a sair na seleção (só 19 casos, p = 0,21) | Pouca informação; uma coluna a mais | 4.3 | Não criado | G2 | ☑ Não criar: 19 casos, p = 0,21 (10/11) |
| D7 | `TotalCharges` fora de ±20% da regra (47 + 9) | a) manter; b) investigar; c) excluir | a | Mostra que o preenchimento de D2 é uma aproximação | 4.2 | Mantidos | Carlos: o preço muda com plano e promoções, e não há histórico de preços (respondido) | ☑ Manter (10/11) |
| D8 | Quebras de linha no `CustomerFeedback` (85 + 15) | Remover ou manter | Remover, se o texto for usado | Só importa se o texto entrar na análise ou no modelo | 4.2 | Aplicado | D9 | ☑ Trocar a quebra de linha por espaço (10/11) |
| D9 | `CustomerFeedback` no modelo | a) com; b) sem; c) os dois | c, com o modelo sem texto como principal | Possível vazamento: uma regra sobre o texto acerta 98,9% | Célula 2 (seção 1); 7.3 do 03 | Fora do 02 (modelo) | G1; Carlos: feedback antes ou depois do cancelamento | ☑ Fora da ABT. O texto vai na tabela separada `gold_churn_texto`, para o modelo com texto (contraexemplo); o modelo oficial é sem texto (10/11) |
| D10 | Serviços com 3 categorias ou sim/não | a) manter 3; b) binário | Testar as duas | "No internet service" repete `InternetService` em 6 colunas: redundância | 3 | Fora do 02 (notebook 04) | G4 | ◐ Sugestão: serviços como sim/não (`_bin`), sempre junto com `InternetService`. A ABT traz as duas versões; seleção final na modelagem (10/11) |
| D11 | Fonte e schema do grupo | a) CSVs de `dados/`; b) tabelas `bronze_*` em `terra_signal` | b | Todos os notebooks; os números precisam bater entre os integrantes | 1.2, 2 | Aplicado: o 02 lê das tabelas `bronze_*` | G3; Carlos: CSV com quebra de linha, schema | ☑ Tabelas `bronze_*` em `workspace.terra_signal`; ignorar (sem apagar) as do upload em `workspace.default` (10/11) |
| D12 | `MonthlyIncome` no modelo | a) manter como candidata; b) ignorar | a, e deixar a seleção decidir | Base externa, sem garantia de precisão; os organizadores dizem que pode ser ignorada. Na exploratória, a renda não muda o risco | 3; seção 10 do 03 | Convertida para número e mantida | Carlos: renda familiar em R$, base externa (respondido) | ◐ Sugestão: descartar (IV abaixo de 0,02, base externa). Fica na ABT; seleção final na modelagem (10/11) |
| D13 | Linhas do `Prediction.csv` | a) 1.409 (clientes); b) 1.461 (linhas físicas) | a | Arquivo com o número errado de linhas pode invalidar a entrega | 1.2 | Fora do 02 (entrega) | Organização: confirmar | ☑ 1.409, uma por `customerID`; a confirmar com a organização (10/11) |
| D14 | Formato da gold | — | `gold_churn_treino` (5.634, com `Churn` 1/0) e `gold_churn_inferencia` (1.409, sem `Churn`), mesmas colunas e ordem; números como número, categorias como texto; comentário em cada coluna; `gold_churn_previsoes` depois do modelo | É o que a modelagem usa; formato estável evita retrabalho | — | Fora do 02 (notebook 04) | — | ☑ ABT com todas as candidatas (originais limpas, serviços nas duas versões, indicador e derivadas) + dicionário com a sugestão de cada coluna; texto em `gold_churn_texto` (10/11) |
| D15 | Outras variáveis sem poder preditivo | a) descartar já; b) deixar a seleção decidir | a: `gender`, `PhoneService` e `MultipleLines` | IV abaixo de 0,02; modelo mais simples de explicar; sexo é variável sensível | Seção 7 e 8 do 03 | Fora do 02 (notebook 04) | — | ◐ Sugestão: descartar `gender`, `PhoneService` e `MultipleLines` (IV abaixo de 0,02). Ficam na ABT; seleção final na modelagem (10/11) |
| D16 | `MonthlyCharges` e `TotalCharges` (repetem outras variáveis) | a) manter; b) trocar por derivada; c) deixar a seleção decidir | Decidir no 04, com as derivadas | Repetem `InternetService` e `tenure` (associação ≥ 0,8) | Seção 6 e 8 do 03 | Fora do 02 (notebook 04) | 04 | ◐ Sugestão: manter `MonthlyCharges` (ou a mensalidade relativa ao pacote) e descartar `TotalCharges` (repete `tenure`, ρ 0,89). As três ficam na ABT; seleção final na modelagem (10/11) |
| D17 | Fonte do 03 | a) CSVs; b) tabelas `bronze_*` | b | Coerência com a D11; os números não mudam | — | Fora do 02 (notebook 03) | — | ☑ Ler da bronze (10/11) |
| D18 | Divisão de papéis | — | A preparação entrega a ABT e sugere; a seleção final das variáveis é feita na modelagem | Quem decide a seleção final das variáveis | — | — | — | ☑ Como proposto (10/11) |
| D19 | Seleção de variáveis | a) só as recomendadas; b) recomendadas + seleção para frente por AUC; c) testes estatísticos com correção para múltiplos testes e retirada das duplicatas | c, conferida com b | É o que entra no modelo | — | — | Notebook 05 | ☑ As 41 colunas do notebook 05 (seções 2 e 3): saem 6 categóricas redundantes e 12 variáveis (duplicatas exatas, versões repetidas do tempo de casa, `gender` sem associação após a correção FDR). A conferência com os métodos da aula (11 variáveis, AUC 0,841) cabe dentro das 41 (11/11) |
| D20 | Segmentação de clientes | a) sem segmentação; b) K-means no comportamento, sem o churn, cruzado com risco × valor | b | Diz que tipo de cliente está em risco e que oferta faz sentido | — | — | Notebook 07 | ◐ Sugestão para o negócio e a apresentação (11/11) |
| D21 | Modelo final | a) logística simples; b) regularização (Lasso, Ridge, Elastic Net); c) árvores (*random forest*, XGBoost, CatBoost) | O de melhor desempenho e mais consistente entre treino, teste e base total | É o modelo que vai para a entrega | — | — | Notebook 06 | ☑ Notebook 06 (modelo do grupo): logística Lasso, 41 colunas, C = 0,3, classes balanceadas, vencedora pela regra de consistência (Ridge a 0,003); AUC 0,830 no teste. Atenção: o balanceamento infla a probabilidade (média de 41% para 27% de churn real) (11/11) |
| D22 | Ponto de corte do Sim/Não | a) 0,5; b) o que maximiza o F1; c) pelo custo da ação | b | Define quantos clientes viram "Yes" | — | — | Notebook 06 | ☑ 0,5046: maior F1 nas probabilidades fora da dobra do treino (notebook 06) (11/11) |
| D23 | Arquivo da entrega | — | 1.409 linhas, `customerID` e `prediction` (Yes/No), na ordem do `inference.csv`, no formato do notebook oficial | Arquivo fora do formato pode invalidar a entrega | — | — | D13; notebook 08 | ☑ `prediction_grupo4_<data_hora>.csv` gerado no 08 e conferido nas 8 regras: 571 de 1.409 com Yes (40,5%); falta comparar os nomes das colunas com o notebook oficial (11/11) |
| D24 | Monitoramento do modelo | a) sem monitoramento; b) PSI das variáveis e das notas, taxa de Yes e reprodução pelo MLflow, com semáforo e app | b | Avisa quando o modelo envelhece, antes de o resultado de negócio mostrar | — | — | Notebook 09 | ◐ Proposta: notebook 09 e app em Streamlit (11/11) |
| D25 | Motor de atendimento | a) só a fila de retenção; b) a nota, os motivos e a oferta de cada cliente na hora da ligação, e a comparação de pacotes para quem ainda não é cliente | b | Leva o modelo para quem fala com o cliente | — | — | Notebook 10 | ◐ Proposta: notebook 10 e página Atendimento do app (12/11) |

## 1. Gerar e conferir o arquivo

> **E1 — O arquivo está no formato que a organização pede?**

**O que faço:** leio as previsões do 06, coloco na ordem do `inference.csv`, troco 1/0 por Yes/No e gravo o arquivo.
Depois releio o arquivo do disco e confiro cada regra. Não pontuo de novo: a previsão é a do pipeline que o 06 validou
e registrou no MLflow.

In [ ]:
# 1.1 Gera o arquivo de previsão na raiz do repositório
from datetime import datetime

# parâmetros do formato: conferir com o NOTEBOOK.ipynb oficial antes de enviar
EQUIPE = "grupo4"
COLUNA_ID, COLUNA_PREVISAO = "customerID", "prediction"
ROTULOS = {1: "Yes", 0: "No"}
COM_INDICE = False

assert "FQ" in globals(), "rode antes a célula %run ./_comum (ela define FQ, raiz e inf_raw)"
scored = spark.table(f"{FQ}.gold_churn_inferencia_scored").toPandas()
ordem = inf_raw["customerID"].str.strip()                    # a ordem oficial do inference.csv

entrega = (pd.DataFrame({COLUNA_ID: ordem})
           .merge(scored[["customerID", "Churn_Previsto", "Probabilidade_Churn"]], left_on=COLUNA_ID,
                  right_on="customerID", how="left", validate="one_to_one"))
entrega[COLUNA_PREVISAO] = entrega["Churn_Previsto"].map(ROTULOS)
arquivo = entrega[[COLUNA_ID, COLUNA_PREVISAO]]

carimbo = datetime.now().strftime("%Y%m%d_%H%M%S")
caminho = os.path.join(raiz, f"prediction_{EQUIPE}_{carimbo}.csv")
arquivo.to_csv(caminho, index=COM_INDICE)
print("Arquivo gravado:", caminho)
display(arquivo.head(10))

In [ ]:
# 1.2 Conferência do arquivo, relido do disco: cada regra da entrega (D13 e D23)
relido = pd.read_csv(caminho, dtype=str, keep_default_na=False)
if COM_INDICE:
    relido = relido.iloc[:, 1:]
testes = [
    ("1.409 linhas, uma por cliente (D13)", len(relido) == 1409),
    (f"colunas {COLUNA_ID} e {COLUNA_PREVISAO}, nessa ordem", relido.columns.tolist() == [COLUNA_ID, COLUNA_PREVISAO]),
    ("customerID sem repetição", relido[COLUNA_ID].is_unique),
    ("os mesmos clientes do inference.csv", set(relido[COLUNA_ID]) == set(ordem)),
    ("na mesma ordem do inference.csv", relido[COLUNA_ID].tolist() == ordem.tolist()),
    ("todo cliente tem previsão", relido[COLUNA_PREVISAO].ne("").all()),
    ("só Yes e No", set(relido[COLUNA_PREVISAO]) <= {"Yes", "No"}),
    ("bate com a tabela do 06", (entrega[COLUNA_PREVISAO].to_numpy() == relido[COLUNA_PREVISAO].to_numpy()).all()),
]
resultado = pd.DataFrame(testes, columns=["regra", "ok"])
display(resultado)
assert resultado["ok"].all(), "arquivo fora do formato. Falhou: " + ", ".join(resultado.loc[~resultado["ok"], "regra"])

n_sim = (relido[COLUNA_PREVISAO] == "Yes").sum()
print(f"Arquivo conferido: {virg(n_sim, 0)} de {virg(len(relido), 0)} clientes com Yes ({virg(100 * n_sim / len(relido), 1)}%).")

> **Como ler a conferência e resposta (E1)**
> - **Cada linha é uma regra da entrega.** O arquivo é relido do disco, para conferir o que de fato vai ser enviado, e
>   não a tabela na memória.
> - **Taxa de Yes:** fica perto de 40%, acima dos 27% de churn do histórico. Não é erro: o limiar do 06 maximiza o
>   F1, e o modelo foi treinado com as classes balanceadas, o que favorece o recall (pegar mais de quem sai, ao custo de
>   alguns alarmes falsos). Na avaliação do 06, isso dá recall perto de 80% e precisão perto de 50%.
> - **E1 — O arquivo está no formato pedido?** Sim, pelas regras acima. Falta só conferir com o `NOTEBOOK.ipynb` oficial
>   os nomes das colunas e o Yes/No (parâmetros da 1.1).
> - **Para enviar:** o arquivo está na raiz do repositório. Fazer o *commit* e o *push* pela pasta Git do Databricks e
>   enviar o link do repositório no formulário da organização.

## 2. Resumo para a apresentação

> **E2 — Que história os números contam, do dado à ação?**

In [ ]:
# 2.1 Os números da história, do dado à ação, lidos das tabelas que os notebooks gravaram
abt = spark.table(f"{FQ}.gold_churn_abt_scored").toPandas()
info = spark.table(f"{FQ}.gold_churn_modelo_info").toPandas().iloc[0]
teste = abt[abt["Base_Modelagem"] == "Teste"]
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score
y_t, p_t = teste["Churn"].astype(int), teste["Probabilidade_Churn"]
prev_t = teste["Churn_Previsto"].astype(int)
d1_2 = teste.sort_values("Probabilidade_Churn", ascending=False).head(int(round(0.2 * len(teste))))

linhas = [
    ("Dados", "Clientes no histórico / a pontuar", f"{virg(len(abt), 0)} / {virg(len(scored), 0)}", "01 e 02"),
    ("Dados", "Taxa de churn no histórico", f"{virg(100 * abt['Churn'].mean(), 1)}%", "03"),
    ("Variáveis", "Colunas no modelo", "41", "05"),
    ("Modelo", "Algoritmo e limiar", f"{info['modelo']}, limiar de {virg(info['limiar'], 4)}", "06"),
    ("Modelo", "ROC-AUC no teste (clientes que o modelo não viu)", virg(roc_auc_score(y_t, p_t), 3), "06"),
    ("Modelo", "Recall / precisão / F1 no teste", f"{virg(100 * recall_score(y_t, prev_t), 0)}% / "
                                                  f"{virg(100 * precision_score(y_t, prev_t), 0)}% / {virg(f1_score(y_t, prev_t), 3)}", "06"),
    ("Ação", "Churn real nos 20% de maior risco do teste", f"{virg(100 * d1_2['Churn'].mean(), 0)}% "
                                                         f"({virg(d1_2['Churn'].sum() / y_t.sum() * 100, 0)}% de todos os que saíram)", "06"),
    ("Ação", "Clientes da inferência previstos como Yes", f"{virg(n_sim, 0)} ({virg(100 * n_sim / len(relido), 1)}%)", "08"),
]
try:
    perfil = spark.table(f"{FQ}.gold_churn_segmentos_perfil").toPandas()
    print("Segmentos (07):")
    display(perfil[["segmento", "nome", "clientes", "pct_carteira", "pct_receita", "churn_pct", "pct_dos_que_sairam"]].round(1))
    linhas.append(("Ação", "Segmentos de clientes", f"{len(perfil)} (detalhe na tabela acima)", "07"))
except Exception as erro:
    print("gold_churn_segmentos_perfil não encontrada: rodar o 07 para incluir os segmentos.", type(erro).__name__)
resumo = pd.DataFrame(linhas, columns=["etapa", "número", "valor", "notebook"])
display(resumo)

> **Como ler e resposta (E2)**
> - **Cada linha é um número da apresentação,** com o notebook de onde vem. Todos são lidos das tabelas gravadas, e
>   não copiados à mão: se algum notebook rodar de novo, o resumo acompanha.
> - **A história em uma frase por etapa:** os dados tinham erros de padronização e ausências, tratados com o grupo
>   (01 e 02); contrato mensal, pouco tempo de casa, fibra sem suporte e cheque eletrônico concentram o churn (03);
>   das variáveis da ABT, 41 entram no modelo, e os métodos da aula confirmam as principais (04 e 05); a logística
>   Lasso separa bem quem sai de quem fica em clientes que ela não viu (06); os segmentos dizem que oferta faz sentido
>   para cada grupo em risco (07); e o arquivo de entrega está conferido (08).

## Checklist final do 08

| # | Decisão | Status | Onde |
|---|---|---|---|
| D13 | 1.409 linhas, uma por `customerID` | ☑ Conferido | 1.2 |
| D23 | Arquivo da entrega | ☑ Gerado e conferido nas 8 regras (11/11); falta só comparar os nomes das colunas com o `NOTEBOOK.ipynb` oficial | 1.1 e 1.2 |

**Saída do 08:** `prediction_grupo4_<timestamp>.csv` na raiz do repositório e o resumo da apresentação (2.1).